In [2]:
function CreateCyclicGroups(prime)
  return CyclicGroup(prime-1), CyclicGroup(prime), prime*(prime-1);
end function;

function WreathProductPolynomial(prime)
  K<z> := CyclotomicField(prime);
  KTs<[T]>:= PolynomialRing(K, prime-1);
  KXTs<X> := PolynomialRing(KTs);
  g := &*[ (X^prime - &+[KTs.i*z^(j*(i-1)) : i in [1..prime-1]]) : j in [1..prime-1]];
  return g;
end function;

// vals should look like [t,1,1,1], [1,t,1,1], etc.
function SpecializePolynomial(g, vals)
  KXTs<X> := Parent(g);
  KTs<[T]>:= BaseRing(KXTs);
  QQt<t> := FunctionField(QQ);
  QQxt<x> := PolynomialRing(QQt);
  mp_cs := hom< KTs -> QQt | vals >;
  mp := hom< KXTs -> QQxt | mp_cs, [x] >;
  return mp(g);
end function;

function ReducePolynomial(gev, ll)
  QQt<t> := FunctionField(QQ);
  QQxt<x> := PolynomialRing(QQt);
  Fp := ResidueClassField(Integers(), ideal<Integers() | ll>);
  FFu<u> := FunctionField(Fp);
  FFxt<xx>:= ChangeRing(QQxt,FFu);
  pi := hom< QQxt -> FFxt | FFxt.1 >;
  gev_FF := pi(gev);
  return gev_FF;
end function;

function FindGaloisSubfield(gev_FF,prime)
  G, roots, data := GaloisGroup(gev_FF);
  //printf "G is %o\n", G;
  assert #G eq prime^(prime-1)*(prime-1);
  C1, C2, prod := CreateCyclicGroups(prime);
  P := DirectProduct(C1,C2);
  Hs := [];
  for el in Subgroups(G : OrderEqual := prod) do
    if IsIsomorphic(el`subgroup,P) then
      Append(~Hs, el);
    end if;
  end for;
  assert #Hs eq 1;
  H := Hs[1];
  //printf "H is %o\n", H;
  fH, hH := GaloisSubgroup(data,H`subgroup);
  return fH, hH;
end function;

function CRTdata(polys)
    moduli := [Characteristic(Parent(poly)) : poly in polys];
    M := &*[m : m in moduli];
    listofMi := [<M/pi, pi> : pi in moduli];
    listofai := [Integers()!((GF(pair[2])!(pair[1]))^(-1)) : pair in listofMi];
    return M, listofMi, listofai;
end function;

function liftUnivariatePoly(h, modM)
    ll := Characteristic(Parent(h));
    lift := hom<GF(ll)->modM|>;
    if h eq 0 then
        return modM!0;
    end if;
    return lift(h);
end function;

function liftPoly(h, S)
    //h has coeffs that are in GF(p)[u]
    coeffs := Coefficients(h);
    modM := BaseRing(S);
    res := &+[ S.1^(i-1) * liftUnivariatePoly(c, modM) : i->c in coeffs];
    return res;
end function;

function LiftBalancedUnivariatePolynomial(pol)
    M := Characteristic(BaseRing(pol));
    M2 := M div 2;
    function lift_balanced(elt)
        elt := Integers()!elt;
        if elt gt M2 then
            elt -:= M;
        end if;
        return elt;
    end function;
    return PolynomialRing(Integers())![lift_balanced(elt) : elt in Coefficients(pol)];
end function;


function createPrimeList(evalpolys, ll, B) 
    listGoodPrimes := [];
    gevFF := AssociativeArray();
    while #listGoodPrimes lt B do
        ll := NextPrime(ll);
        is_good_prime := true;
        gev_FF_list := [];
        for gev in evalpolys do 
            gev_FF := ReducePolynomial(gev,ll);
            Append(~gev_FF_list, gev_FF);
            if not IsIrreducible(gev_FF) then
                is_good_prime := false;
                break;
            end if;
        end for;
        if is_good_prime then
            Append(~listGoodPrimes, ll);
            gevFF[ll] := gev_FF_list;
        end if;
    end while;
    return listGoodPrimes, gevFF;
end function;



In [3]:
function NewSpecializePolynomial(g, vals, ll)
  R<X> := Parent(g);
  S<[T]>:= BaseRing(R);
  S1<t> := FunctionField(GF(ll));
  R1<x> := PolynomialRing(S1);
  mp_cs := hom< S -> S1 | vals >;
  mp := hom< R -> R1 | mp_cs, [x] >;
  return mp(g);
end function;

function createPointList(g_red, ll, N)
    FFt<t> := FunctionField(GF(ll));
    listGoodPoints := [];
    gevFF := AssociativeArray();
    while #listGoodPoints lt N do
        pt := [ Random(GF(ll))*t + Random(GF(ll)) : i in [1..Rank(BaseRing(Parent(g_red)))]];
        gev_FF := NewSpecializePolynomial(g_red, pt, ll);
        print gev_FF;
        if IsIrreducible(gev_FF) then
            Append(~listGoodPoints, pt);
            gevFF[pt] := gev_FF;
            print "pt added";
        else
            print "pt not added";
        end if;
    end while;
    return listGoodPoints, gevFF;
end function;

In [7]:

SetVerbose("GaloisGroup",0);
p := 3;
C1, C2 := CreateCyclicGroups(p);
g := WreathProductPolynomial(p);
KXTs<X> := Parent(g);
KTs<[T]>:= BaseRing(KXTs);

function NewReducePolynomial(g, ll)
    R := Parent(g);
    S := BaseRing(R);
    SFF := ChangeRing(S, GF(ll));
    RFF := ChangeRing(R, SFF);
    homS := hom<S->SFF | [SFF.i : i in [1..Rank(S)]]>;
    homR := hom<R->RFF | homS, RFF.1>;
    gFF := homR(g);
    return gFF;
end function;

start_ll := 40;
while true do
    start_ll := NextPrime(start_ll);
    g_red := NewReducePolynomial(g, start_ll); 
    ptList, gpolys := createPointList(g_red, start_ll, 10);
    print "ptList found";
    polys := [];
    for pt in ptList do
        print pt, "start";
        time fH, _ := FindGaloisSubfield(gpolys[pt],p); 
        Append(~polys, fH);
        print pt, "done";
    end for;
    break;
end while;

x^6 + (23*t + 37)*x^3 + 2*t^2 + 31*t + 30
pt added
x^6 + (38*t + 33)*x^3 + 22*t^2 + 33*t + 33
pt added
x^6 + (22*t + 34)*x^3 + 32*t^2 + 36*t + 16
pt added
x^6 + (8*t + 8)*x^3 + 33*t^2 + 33*t + 13
pt added
x^6 + (23*t + 2)*x^3 + 28*t^2 + 35*t + 39
pt added
x^6 + (15*t + 34)*x^3 + 39*t^2 + 11*t + 37
pt added
x^6 + (22*t + 31)*x^3 + 22*t^2 + 27*t + 39
pt added
x^6 + (31*t + 29)*x^3 + 22*t^2 + 10*t + 19
pt added
x^6 + (9*t + 2)*x^3 + 36*t^2 + 9*t + 1
pt added
x^6 + (13*t + 5)*x^3 + 4*t^2 + 19*t + 34
pt added
ptList found
[
8*t + 37,
39*t + 29
]
start
Time: 0.050
[
8*t + 37,
39*t + 29
]
done
[
38*t + 26,
32*t + 3
]
start
Time: 0.060
[
38*t + 26,
32*t + 3
]
done
[
25*t + 38,
31*t + 28
]
start
Time: 0.050
[
25*t + 38,
31*t + 28
]
done
[
18*t + 28,
3*t + 23
]
start
Time: 0.060
[
18*t + 28,
3*t + 23
]
done
[
32*t + 8,
5*t + 18
]
start
Time: 0.050
[
32*t + 8,
5*t + 18
]
done
[
18*t + 35,
10*t + 22
]
start
Time: 0.070
[
18*t + 35,
10*t + 22
]
done
[
37*t + 19,
14*t + 28
]
start
Time: 0.060
[
37*t

In [8]:
function MultivariateInterpolation(R, points, values)
    assert #points eq #values;
    assert #values gt 0;
    n := Rank(R);
    F := BaseRing(R);
    monomials := [R|];
    degree := 0;
    while #points ge #monomials + Binomial(degree + n -1, degree) do
        monomials cat:= SetToSequence(MonomialsOfDegree(R, degree));
        degree +:= 1;
    end while;
    mat := Matrix(F, [[Evaluate(m, p) : p in points] : m in monomials]);
    v := Vector(F, values);
    s := Solution(mat, v);
    return &+[s[i]*m : i->m in monomials];

end function;

In [67]:
#{Monomials(p) : p in polys};

1


In [9]:
ptList2 := &cat[[ [Evaluate(x, t0) : x in pt] : t0 in [1..100]] : pt in ptList];
polys2 := &cat[[Evaluate(Coefficients(poly)[1], t0) : t0 in [1..100]] : poly in polys];

In [11]:
MultivariateInterpolation(PolynomialRing(Parent(ptList2[1][1]),2), ptList2, polys2);

11*$.1^2 + 30*$.1*$.2 + 11*$.2^42


In [12]:
GF(41)!216;

11


In [6]:
gev_FF[1073741827]

[
xx^20 + (1073741823*u + 3)*xx^15 + (6*u^2 + 1073741818*u + 4)*xx^10 + (1073741823*u^3 + 9*u^2 + 1073741819*u + 2)*xx^5 + u^4 + 1073741824*u^3 + 4*u^2 + 1073741825*u + 1,
xx^20 + (u + 1073741825)*xx^15 + (u^2 + u + 1073741826)*xx^10 + (u^3 + 1073741826*u^2 + 1073741824*u + 2)*xx^5 + u^4 + 1073741824*u^3 + 4*u^2 + 1073741825*u + 1,
xx^20 + (u + 1073741825)*xx^15 + (u^2 + 1073741823*u + 4)*xx^10 + (u^3 + 1073741821*u^2 + 7*u + 1073741824)*xx^5 + u^4 + 1073741824*u^3 + 4*u^2 + 1073741825*u + 1,
xx^20 + (u + 1073741825)*xx^15 + (u^2 + 1073741823*u + 4)*xx^10 + (u^3 + 1073741826*u^2 + 2*u + 1073741824)*xx^5 + u^4 + 1073741824*u^3 + 4*u^2 + 1073741825*u + 1
]


In [7]:
//polys := [* *]

In [8]:
for ll in ourPrimes do
    for gpoly in gev_FF[ll] do
        time fH, _ := FindGaloisSubfield(gpoly,p); 
        Append(~polys, fH);
    end for;
    printf "done with ll = %o\n", ll;
end for;

Time: 65.590
Time: 83.190
Time: 56.040
Time: 51.880
done with ll = 1073741827
Time: 59.610
Time: 43.270
Time: 52.280
Time: 48.280
done with ll = 1073741833
Time: 53.120
Time: 78.350
Time: 84.500
Time: 85.690
done with ll = 1073741843
Time: 63.180
Time: 47.070
Time: 61.210
Time: 93.470
done with ll = 1073741857
Time: 87.110
Time: 62.140
Time: 74.360
Time: 88.530
done with ll = 1073741953
Time: 91.580
Time: 255.530
Time: 59.730
Time: 82.550
done with ll = 1073741987
Time: 60.420
Time: 90.340
Time: 89.270
Time: 98.050
done with ll = 1073741993
Time: 50.100
Time: 61.280
Time: 58.500
Time: 62.870
done with ll = 1073742037
Time: 55.710
Time: 82.280
Time: 83.540
Time: 64.370
done with ll = 1073742053
Time: 88.860
Time: 93.850
Time: 83.900
Time: 58.480
done with ll = 1073742073


In [20]:
mypolys := polys[1..8];
M, listofMi, listofai := CRTdata(mypolys);
modM := Integers(M);
S<XX> := PolynomialRing(modM);
res := [S|];
for i->pol in mypolys do
    ai := listofai[i];
    Mi := listofMi[i][1];
    pi := listofMi[i][2];
    c := GF(pi)!(10)^(-1); //scale by inverse of 10?
    coeffsmodp := Coefficients(pol*ai);
    if #coeffsmodp gt #res then
        res cat:=[S| 0 :
        _ in [#res+1..#coeffsmodp]];
    end if;
    for j->coeff in coeffsmodp do
        res[j] +:= liftPoly(coeff, S)* Integers(M)!Mi;
    end for;
end for;

res := [LiftBalancedUnivariatePolynomial(res[i]) : i in [1 .. #res]];

In [21]:
res eq big_res

false


In [279]:
q1 := listofMi[1][2];
M1 := listofMi[1][1];
a1 := Integers()!(1/GF(q1)!(M1));
!(a1*M1);

1


In [231]:
LiftBalancedUnivariatePolynomial(res[1]);

321753715851328921796683846136633804222658651315432594900694994764446953543557242922825888010327430582635676254885616989323945043832710


In [194]:
foo<Z> := Parent(res[1]);
LiftBalancedUnivariatePolynomial(-Z);

-XX


In [197]:
LiftBalancedUnivariatePolynomial( res[2])

-62810030454628723502609058703735150380069937775798919956612853609141546698562414767075876267352125476855896019836673172722196373506591*XX^98 - 317401497954096899031321740123038632893549389997168103836901717860006406858086228702569021837608647583362624121494486306888953373202752*XX^97 + 35573162499628460748438964827999607078982664941491192583248284556342216237369485886023727796945846451012600398220348618093707645817866*XX^96 + 235675574193151653799464890427115526252611443965946283347336624135831899821350229019143247216973035395684091406979642877930776157941511*XX^95 + 342705015895639147333847347670802139692479945040753163939536809190347962067935343077472141137167732238045832495824834043612300610578570*XX^94 - 151931530672251742249391338711324393080775884030587802058188643570697766892006820626377399652894111537636888542915214629031814244345011*XX^93 - 274370224755355700072751526968805229201838235377974231161253047653674922180335007304469882537055095368210546504487618099825538600548494*X

In [195]:
LiftBalancedUnivariatePolynomial( res[2])

201003756133701257288544720066072122934542833856619168567106126767706979836074497898665682886875930989640858571036595474738844905556873*XX^98 - 170208571123001145228241300744256294447785557626540490189751336637464689058458277455177753299926979070813569668983065002494515251052657*XX^97 - 360734519098737037218295813197541426708605613615518835559785305303109000225635522188575278021845022675102585018787116419025390600509999*XX^96 + 105339735525471868039162072584181816453220286490634121383773465111156185582499269277595351656509480442831237126790856288004626813060885*XX^95 + 209198282721214361056006618938247566889091933393056397481061286064505866772981921294724464749895318290669467664963787244777559919063797*XX^94 + 291263741977796210298904395521425792824276732801927877902799010735212718738034024114229785082435906501987843518128202466177764430071515*XX^93 - 270167880626774896330432979497713914792296298236077152339026056737436767161277891615461210857041207908618803289307455360549555591246654*

In [133]:
res[1]

5139409731*XX^100 + 7449442452*XX^99 + 8351903415*XX^98 + 8371398923*XX^97 + 4076634058*XX^96 + 7403532666*XX^95 + 4197019222*XX^94 + 7735928456*XX^93 + 7259992546*XX^92 + 3881069634*XX^91 + 5993635322*XX^90 + 5442674436*XX^89 + 10902942418*XX^88 + 4701818498*XX^87 + 6619459425*XX^86 + 8098860122*XX^85 + 5703854047*XX^84 + 4179888509*XX^83 + 4804659667*XX^82 + 8118469130*XX^81 + 8399764382*XX^80 + 7884988977*XX^79 + 5853202187*XX^78 + 4755338608*XX^77 + 8455994642*XX^76 + 8645112101*XX^75 + 7466587074*XX^74 + 7609553605*XX^73 + 4221384612*XX^72 + 6610743420*XX^71 + 9312659821*XX^70 + 7321755788*XX^69 + 7230223578*XX^68 + 5098811763*XX^67 + 4829211968*XX^66 + 5760158699*XX^65 + 9989346596*XX^64 + 7954243949*XX^63 + 6545833520*XX^62 + 8547651212*XX^61 + 9034439523*XX^60 + 2658465959*XX^59 + 5427892085*XX^58 + 5460386092*XX^57 + 6933783481*XX^56 + 4905565511*XX^55 + 4499479846*XX^54 + 4513971516*XX^53 + 5878215942*XX^52 + 2888002594*XX^51 + 7208657575*XX^50 + 6219267074*XX^49 + 2955420756

In [127]:

LiftBalancedUnivariatePolynomial(res[1])

5139409731*XX^100 + 7449442452*XX^99 + 8351903415*XX^98 + 8371398923*XX^97 + 4076634058*XX^96 + 7403532666*XX^95 + 4197019222*XX^94 + 7735928456*XX^93 + 7259992546*XX^92 + 3881069634*XX^91 + 5993635322*XX^90 + 5442674436*XX^89 + 10902942418*XX^88 + 4701818498*XX^87 + 6619459425*XX^86 + 8098860122*XX^85 + 5703854047*XX^84 + 4179888509*XX^83 + 4804659667*XX^82 + 8118469130*XX^81 + 8399764382*XX^80 + 7884988977*XX^79 + 5853202187*XX^78 + 4755338608*XX^77 + 8455994642*XX^76 + 8645112101*XX^75 + 7466587074*XX^74 + 7609553605*XX^73 + 4221384612*XX^72 + 6610743420*XX^71 + 9312659821*XX^70 + 7321755788*XX^69 + 7230223578*XX^68 + 5098811763*XX^67 + 4829211968*XX^66 + 5760158699*XX^65 + 9989346596*XX^64 + 7954243949*XX^63 + 6545833520*XX^62 + 8547651212*XX^61 + 9034439523*XX^60 + 2658465959*XX^59 + 5427892085*XX^58 + 5460386092*XX^57 + 6933783481*XX^56 + 4905565511*XX^55 + 4499479846*XX^54 + 4513971516*XX^53 + 5878215942*XX^52 + 2888002594*XX^51 + 7208657575*XX^50 + 6219267074*XX^49 + 2955420756

In [132]:
res[10]

4849887718*XX^92 + 8172686128*XX^91 + 3651772960*XX^90 + 6468144593*XX^89 + 5606898564*XX^88 + 3944417022*XX^87 + 6022243011*XX^86 + 6108297317*XX^85 + 5641856165*XX^84 + 7312458564*XX^83 + 9098944776*XX^82 + 4262741914*XX^81 + 7908278575*XX^80 + 9461998519*XX^79 + 8665037814*XX^78 + 5407613040*XX^77 + 7823590847*XX^76 + 5718463256*XX^75 + 5905405175*XX^74 + 7443230896*XX^73 + 4704562347*XX^72 + 5615644717*XX^71 + 6859960334*XX^70 + 9143443855*XX^69 + 5639862163*XX^68 + 3556135932*XX^67 + 6199169412*XX^66 + 3298936527*XX^65 + 2882622056*XX^64 + 4217539301*XX^63 + 9179051710*XX^62 + 3363602137*XX^61 + 8670911365*XX^60 + 7820478416*XX^59 + 5178924313*XX^58 + 5966336001*XX^57 + 3773108412*XX^56 + 7446540946*XX^55 + 2662638363*XX^54 + 3861057925*XX^53 + 8305624993*XX^52 + 2195477304*XX^51 + 6830366444*XX^50 + 4825242569*XX^49 + 4892584731*XX^48 + 6981269864*XX^47 + 5329677877*XX^46 + 7585869521*XX^45 + 5107910265*XX^44 + 7624217775*XX^43 + 6469563302*XX^42 + 5078628677*XX^41 + 5161656351*X